# NB-06: RQ2 — Lead-Lag by Weapon Class

**The Peacekeepers' Arms Race — Stability-Instability Paradox**

Tests whether acquisitions of specific weapon classes precede increases in low-intensity conflict within a multi-year window. Treatment = SIPRI TIV deliveries by weapon class; outcome = UCDP conflict participation counts.

**Three-stage analysis:**
1. Cross-correlation functions (CCF) at lags 0–5 years — descriptive lead-lag pattern
2. Dumitrescu-Hurlin (2012) panel Granger non-causality test — inferential
3. Placebo test (within-country shuffle) — falsification

**Predictions under the Drone Effect:**
- AIR class (aircraft + drones) Granger-precedes `part_n_minor` at 1–2 year lag
- AIR class does *not* Granger-precede `part_n_war` at any lag
- MISSILES class shows a similar pattern (precision strike → low-intensity)
- NAVAL and GROUND show no clean lead-lag (older technology layers)

Operating granularity: **annual** (treatment is yearly TIV deliveries; outcome is annual participation count). Monthly granularity is feasible via `rq2_outcome_monthly.parquet` but kept as an optional extension.

## Section 0 — Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import norm, spearmanr
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

from src.io_utils import load_checkpoint, save_checkpoint
from src.config import CLEAN_DIR, FIGURES_DIR, TABLES_DIR, SEED

np.random.seed(SEED)  # for placebo shuffle reproducibility

plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
sns.set_palette("tab10")

FIG_DIR = FIGURES_DIR / "nb06"
TBL_DIR = TABLES_DIR / "nb06"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR.mkdir(parents=True, exist_ok=True)

# Load panels
rq2_treat = load_checkpoint(CLEAN_DIR / "rq2_treatment_annual.parquet")
master    = load_checkpoint(CLEAN_DIR / "master_panel.parquet")

print(f"rq2_treatment_annual: {rq2_treat.shape}")
print(f"master_panel:         {master.shape}")

# Strategic weapon-class grouping (4 classes from 11 SIPRI categories)
WEAPON_CLASSES = {
    "AIR":      ["tiv_cat_aircraft"],
    "MISSILES": ["tiv_cat_missiles", "tiv_cat_air_defence_systems"],
    "NAVAL":    ["tiv_cat_ships", "tiv_cat_naval_weapons"],
    "GROUND":   ["tiv_cat_armoured_vehicles", "tiv_cat_artillery"],
}

OUTCOMES = [
    "part_n_minor",            # predicted: AIR/MISSILES → positive lead
    "part_n_war",              # predicted: no lead
    "part_n_extraterritorial", # predicted: AIR/MISSILES → positive lead
]

ANALYSIS_YEAR_MIN = 1989
ANALYSIS_YEAR_MAX = 2024
MAX_LAG = 5
MIN_OBS_PER_COUNTRY = 12  # need enough obs after differencing + lagging

print(f"\nWeapon classes: {list(WEAPON_CLASSES)}")
print(f"Outcomes:       {OUTCOMES}")
print(f"Year window:    {ANALYSIS_YEAR_MIN}–{ANALYSIS_YEAR_MAX}")

[checkpoint] loaded ← rq2_treatment_annual.parquet  (6,912 rows)
[checkpoint] loaded ← master_panel.parquet  (15,168 rows)
rq2_treatment_annual: (6912, 18)
master_panel:         (15168, 61)

Weapon classes: ['AIR', 'MISSILES', 'NAVAL', 'GROUND']
Outcomes:       ['part_n_minor', 'part_n_war', 'part_n_extraterritorial']
Year window:    1989–2024


## Section 1 — Build Series and Test Stationarity

Builds the analysis panel: country-year with one column per weapon class (summed within class, log1p'd to dampen variance) and one column per outcome (log1p'd). Then tests stationarity via ADF for a sample of country series — non-stationary series are first-differenced to make Granger results meaningful.

Standard practice: first-difference everything for safety. We test stationarity primarily to report the rate of non-stationarity (it's almost always high for cumulative arms data), not to decide per-country.

In [2]:
# Build the analysis panel
panel = master[
    (master["year"] >= ANALYSIS_YEAR_MIN) & (master["year"] <= ANALYSIS_YEAR_MAX)
][["iso3", "year"] + OUTCOMES + sum(WEAPON_CLASSES.values(), [])].copy()

# Aggregate weapon categories into 4 strategic classes
for class_name, cat_cols in WEAPON_CLASSES.items():
    panel[f"tiv_{class_name}"] = panel[cat_cols].fillna(0).sum(axis=1)
    # log1p to compress the long right tail of TIV deliveries
    panel[f"log_tiv_{class_name}"] = np.log1p(panel[f"tiv_{class_name}"])

# log1p outcomes too (count data, right-skewed)
for outcome in OUTCOMES:
    panel[f"log_{outcome}"] = np.log1p(panel[outcome].fillna(0))

TREATMENTS = [f"log_tiv_{c}" for c in WEAPON_CLASSES]
LOG_OUTCOMES = [f"log_{o}" for o in OUTCOMES]

print(f"Panel shape: {panel.shape}")
print(f"Countries:   {panel['iso3'].nunique()}")
print(f"Year range:  {panel['year'].min()}–{panel['year'].max()}")

# Stationarity audit (ADF test) on a random sample of country series
rng = np.random.default_rng(SEED)
sample_iso3 = rng.choice(panel["iso3"].unique(), size=30, replace=False)

adf_rows = []
for iso3 in sample_iso3:
    sub = panel[panel["iso3"] == iso3].sort_values("year")
    for col in TREATMENTS + LOG_OUTCOMES:
        s = sub[col].dropna()
        if len(s) < 12:
            continue
        if np.var(s) < 1e-6:
            continue
        try:
            stat, pval, *_ = adfuller(s, regression="c", autolag="AIC")
            adf_rows.append({"iso3": iso3, "series": col, "p": pval,
                             "stationary": pval < 0.05})
        except Exception:
            pass

adf_df = pd.DataFrame(adf_rows)
stationary_rate = adf_df.groupby("series")["stationary"].mean().round(2)
print("\n=== ADF stationarity rate by series (30-country sample) ===")
print(stationary_rate.to_string())
print("\n(Low rates → non-stationary; first-differencing is therefore applied to all series below.)")

# First-difference everything within country
panel = panel.sort_values(["iso3", "year"])
for col in TREATMENTS + LOG_OUTCOMES:
    panel[f"d_{col}"] = panel.groupby("iso3")[col].diff()

DIFF_TREATMENTS = [f"d_{t}" for t in TREATMENTS]
DIFF_OUTCOMES   = [f"d_{o}" for o in LOG_OUTCOMES]

print(f"\nDifferenced columns added: {len(DIFF_TREATMENTS + DIFF_OUTCOMES)}")
adf_df.to_csv(TBL_DIR / "section1_stationarity_audit.csv", index=False)

Panel shape: (6912, 23)
Countries:   192
Year range:  1989–2024


d:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2



=== ADF stationarity rate by series (30-country sample) ===
series
log_part_n_extraterritorial    0.27
log_part_n_minor               0.32
log_part_n_war                 0.62
log_tiv_AIR                    0.70
log_tiv_GROUND                 0.68
log_tiv_MISSILES               0.61
log_tiv_NAVAL                  0.77

(Low rates → non-stationary; first-differencing is therefore applied to all series below.)

Differenced columns added: 7


d:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
d:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
d:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2
d:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\.venv\Lib\site-packages\statsmodels\regression\linear_model.py:955: RuntimeWarning: divide by zero encounter

## Section 2 — Cross-Correlation Functions

Descriptive lead-lag. For each (country, weapon class, outcome) trio, computes Pearson correlation between treatment at time *t* and outcome at time *t+k* for k = 0…5 years. Positive *k* means treatment leads outcome — exactly what the Drone Effect hypothesis predicts.

Country-level CCFs are then averaged across the panel to produce a single summary curve per (weapon class, outcome) pair.

In [3]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="numpy")

def cross_correlation(x, y, lag):
    """Pearson correlation of x_t with y_{t+lag}. Positive lag = x leads y."""
    if lag < 0:
        return np.corrcoef(x[-lag:], y[:lag])[0, 1] if len(x) + lag > 2 else np.nan
    if lag > 0:
        return np.corrcoef(x[:-lag], y[lag:])[0, 1] if len(x) - lag > 2 else np.nan
    return np.corrcoef(x, y)[0, 1] if len(x) > 2 else np.nan

ccf_rows = []
for class_name in WEAPON_CLASSES:
    treat_col = f"d_log_tiv_{class_name}"
    for outcome in OUTCOMES:
        out_col = f"d_log_{outcome}"
        for iso3 in panel["iso3"].unique():
            sub = panel[panel["iso3"] == iso3].sort_values("year")
            sub = sub.dropna(subset=[treat_col, out_col])
            if len(sub) < MIN_OBS_PER_COUNTRY:
                continue
            x = sub[treat_col].values
            y = sub[out_col].values
            if np.std(x) == 0 or np.std(y) == 0:
                continue
            for lag in range(0, MAX_LAG + 1):
                rho = cross_correlation(x, y, lag)
                if np.isfinite(rho):
                    ccf_rows.append({
                        "iso3": iso3, "weapon": class_name, "outcome": outcome,
                        "lag": lag, "rho": rho,
                    })

ccf_df = pd.DataFrame(ccf_rows)

# Aggregate: mean CCF across countries by (weapon, outcome, lag)
ccf_mean = (
    ccf_df.groupby(["weapon", "outcome", "lag"])["rho"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .rename(columns={"mean": "rho_mean", "std": "rho_std", "count": "n_countries"})
)

# Pivot for compact display: rows = (weapon, outcome), cols = lag
ccf_pivot = ccf_mean.pivot_table(
    index=["weapon", "outcome"], columns="lag", values="rho_mean"
).round(3)

print("=== Average cross-correlation across countries (lags 0–5) ===\n")
print(ccf_pivot.to_string())

ccf_df.to_csv(TBL_DIR / "section2_ccf_long.csv", index=False)
ccf_mean.to_csv(TBL_DIR / "section2_ccf_aggregated.csv", index=False)

# Print which (weapon, outcome) pairs show peak ρ at lag > 0 (the lead-lag signature)
print("\n=== Peak-lag check (lag > 0 means treatment leads outcome) ===")
for (w, o), grp in ccf_mean.groupby(["weapon", "outcome"]):
    peak = grp.loc[grp["rho_mean"].idxmax()]
    print(f"  {w:9s} → {o:25s}  peak ρ = {peak['rho_mean']:+.3f} at lag {int(peak['lag'])}")

=== Average cross-correlation across countries (lags 0–5) ===

lag                                   0      1      2      3      4      5
weapon   outcome                                                          
AIR      part_n_extraterritorial -0.005  0.007  0.004  0.006  0.015 -0.030
         part_n_minor             0.009  0.004  0.016 -0.002  0.014 -0.025
         part_n_war               0.003  0.014 -0.012  0.007 -0.009  0.004
GROUND   part_n_extraterritorial  0.019 -0.022 -0.017  0.012  0.017 -0.025
         part_n_minor             0.011  0.004 -0.013 -0.010  0.011 -0.003
         part_n_war               0.040 -0.026 -0.009  0.025  0.004 -0.019
MISSILES part_n_extraterritorial -0.047  0.013  0.010 -0.018  0.023  0.008
         part_n_minor            -0.033  0.022 -0.004 -0.016  0.016  0.003
         part_n_war              -0.016  0.034 -0.011  0.003  0.020 -0.027
NAVAL    part_n_extraterritorial -0.009  0.009 -0.009  0.028 -0.011 -0.006
         part_n_minor            -0.0

## Section 3 — Dumitrescu-Hurlin Panel Granger Test

Tests, on the full country panel, whether the lagged values of treatment series x add explanatory power to predicting outcome y beyond y's own lags. Procedure:

1. For each country *i*: run restricted ($y_{i,t} = \alpha_i + \sum_k \gamma_{i,k} y_{i,t-k}$) and unrestricted ($y_{i,t} = \alpha_i + \sum_k \gamma_{i,k} y_{i,t-k} + \sum_k \beta_{i,k} x_{i,t-k}$) OLS; compute F-stat $W_{i,L}$ for the null $\beta_{i,1} = \ldots = \beta_{i,L} = 0$
2. Average across countries: $\bar{W}_L = N^{-1} \sum_i W_{i,L}$
3. Standardize: $Z_L = \sqrt{N/(2L)} \cdot (\bar{W}_L - L) / \sqrt{L}$
4. Under $H_0$: x does not Granger-cause y for any country, $Z_L \overset{d}{\to} \mathcal{N}(0,1)$

We test at lags 1, 2, 3 years. Significant positive Z = treatment Granger-precedes outcome on average across the panel.

*This test is interpreted as predictive precedence, not causation.*

In [4]:
from src.stats_panel import dumitrescu_hurlin

# Run DH for every (weapon class, outcome, lag) cell
dh_rows = []
for class_name in WEAPON_CLASSES:
    treat_col = f"d_log_tiv_{class_name}"
    for outcome in OUTCOMES:
        out_col = f"d_log_{outcome}"
        for lag in [1, 2, 3]:
            res = dumitrescu_hurlin(panel, treat_col, out_col, lag)
            dh_rows.append({
                "weapon":      class_name,
                "outcome":     outcome,
                "lag":         lag,
                "Z":           res["Z"],
                "p":           res["p"],
                "W_bar":       res["W_bar"],
                "n_countries": res["n_countries"],
            })

dh_df = pd.DataFrame(dh_rows)
dh_df["sig"] = pd.cut(dh_df["p"], bins=[0, 0.01, 0.05, 0.10, 1],
                      labels=["***", "**", "*", ""]).astype(str)

# Pivot for display
z_pivot = dh_df.pivot_table(
    index=["weapon", "outcome"], columns="lag", values="Z"
).round(3)
p_pivot = dh_df.pivot_table(
    index=["weapon", "outcome"], columns="lag", values="p"
).round(4)

print("=== Dumitrescu-Hurlin Z-statistic (positive = treatment leads outcome) ===\n")
print(z_pivot.to_string())
print("\n=== Dumitrescu-Hurlin one-sided p-values ===\n")
print(p_pivot.to_string())

dh_df.to_csv(TBL_DIR / "section3_dumitrescu_hurlin.csv", index=False)

# Significant findings only
sig_findings = dh_df[dh_df["p"] < 0.10].sort_values("p")
print(f"\n=== Significant DH results (p < 0.10): {len(sig_findings)} ===")
if len(sig_findings):
    print(sig_findings[["weapon", "outcome", "lag", "Z", "p", "sig", "n_countries"]]
          .to_string(index=False))

=== Dumitrescu-Hurlin Z-statistic (positive = treatment leads outcome) ===

lag                                   1      2      3
weapon   outcome                                     
AIR      part_n_extraterritorial  0.541 -3.617 -5.184
         part_n_minor             0.232 -2.936 -4.824
         part_n_war              -0.050 -2.862 -4.089
GROUND   part_n_extraterritorial  4.927 -2.845 -4.909
         part_n_minor             1.870 -2.920 -4.649
         part_n_war               6.423 -2.307 -4.543
MISSILES part_n_extraterritorial  2.018 -2.619 -3.002
         part_n_minor             3.637 -2.492 -3.732
         part_n_war               2.256 -1.265 -3.816
NAVAL    part_n_extraterritorial  2.040 -2.970 -4.181
         part_n_minor             3.289 -2.091 -3.189
         part_n_war               1.005 -2.538 -4.397

=== Dumitrescu-Hurlin one-sided p-values ===

lag                                    1       2       3
weapon   outcome                                        
AIR    

## Section 4 — Placebo Test (Falsification)

Shuffles the treatment series within each country (preserving country-fixed effects but breaking temporal alignment between treatment and outcome). If the real DH result is genuine, the placebo should fail to detect Granger causation. If the placebo *also* finds significant lead-lag, the real result is spurious (likely a finite-sample artifact or autocorrelation issue).

Run the placebo 50 times and report the fraction of iterations where each (weapon, outcome, lag) cell would be significant at p < 0.05. A well-behaved test rejects the null around 5% of the time on shuffled data.

In [5]:
N_PLACEBO = 50
PLACEBO_ALPHA = 0.05

# ── Step 1: precompute per-country numpy arrays (one-time setup) ────────────────
treat_cols = [f"d_log_tiv_{c}" for c in WEAPON_CLASSES]
outcome_cols = [f"d_log_{o}" for o in OUTCOMES]

country_data = {}
for iso3, g in panel.groupby("iso3", sort=False):
    g = g.sort_values("year")
    country_data[iso3] = {col: g[col].values for col in treat_cols + outcome_cols}

print(f"Pre-extracted arrays for {len(country_data)} countries.")


# ── Step 2: fast DH using numpy lstsq instead of statsmodels OLS ─────────────────
from src.stats_panel import dumitrescu_hurlin_fast as _dh_fast


# ── Step 3: placebo loop (numpy-only inside, no pandas in the hot path) ──────────
rng = np.random.default_rng(SEED)
placebo_rows = []

from itertools import product
total_cells = len(WEAPON_CLASSES) * len(OUTCOMES) * 3
done = 0
for class_name, outcome, lag in product(WEAPON_CLASSES, OUTCOMES, [1, 2, 3]):
    treat_col = f"d_log_tiv_{class_name}"
    out_col   = f"d_log_{outcome}"
    sig_count = 0
    z_values = []
    for trial in range(N_PLACEBO):
        # Shuffle treatment column within each country, in-numpy
        shuffled = {}
        for iso3, arrs in country_data.items():
            new = dict(arrs)
            x = arrs[treat_col].copy()
            valid = np.isfinite(x)
            if valid.sum() > 1:
                x[valid] = rng.permutation(x[valid])
            new[treat_col] = x
            shuffled[iso3] = new
        Z, p, _ = _dh_fast(shuffled, treat_col, out_col, lag)
        if np.isfinite(Z):
            z_values.append(Z)
            if p < PLACEBO_ALPHA:
                sig_count += 1
    placebo_rows.append({
        "weapon":           class_name,
        "outcome":          outcome,
        "lag":              lag,
        "placebo_sig_rate": sig_count / N_PLACEBO,
        "placebo_Z_mean":   np.mean(z_values) if z_values else np.nan,
        "placebo_Z_std":    np.std(z_values) if z_values else np.nan,
    })
    done += 1
    print(f"  [{done:2d}/{total_cells}] {class_name:9s} → {outcome:25s} L={lag}  done")

placebo_df = pd.DataFrame(placebo_rows)

# Merge with real DH for side-by-side comparison
comparison = dh_df[["weapon", "outcome", "lag", "Z", "p"]].merge(
    placebo_df, on=["weapon", "outcome", "lag"]
).rename(columns={"Z": "Z_real", "p": "p_real"})

print("\n=== Real DH vs Placebo comparison ===\n")
print(comparison.round(3).to_string(index=False))

comparison.to_csv(TBL_DIR / "section4_placebo_comparison.csv", index=False)

comparison["trustworthy"] = (
    (comparison["p_real"] < 0.05)
    & (comparison["placebo_sig_rate"] < 0.15)
    & (comparison["Z_real"] > 2 * comparison["placebo_Z_std"].fillna(1))
)
print(f"\n=== Trustworthy findings: {comparison['trustworthy'].sum()} ===")
if comparison["trustworthy"].any():
    print(comparison[comparison["trustworthy"]][
        ["weapon", "outcome", "lag", "Z_real", "p_real", "placebo_sig_rate"]
    ].to_string(index=False))

Pre-extracted arrays for 192 countries.
  [ 1/36] AIR       → part_n_minor              L=1  done
  [ 2/36] AIR       → part_n_minor              L=2  done
  [ 3/36] AIR       → part_n_minor              L=3  done
  [ 4/36] AIR       → part_n_war                L=1  done
  [ 5/36] AIR       → part_n_war                L=2  done
  [ 6/36] AIR       → part_n_war                L=3  done
  [ 7/36] AIR       → part_n_extraterritorial   L=1  done
  [ 8/36] AIR       → part_n_extraterritorial   L=2  done
  [ 9/36] AIR       → part_n_extraterritorial   L=3  done
  [10/36] MISSILES  → part_n_minor              L=1  done
  [11/36] MISSILES  → part_n_minor              L=2  done
  [12/36] MISSILES  → part_n_minor              L=3  done
  [13/36] MISSILES  → part_n_war                L=1  done
  [14/36] MISSILES  → part_n_war                L=2  done
  [15/36] MISSILES  → part_n_war                L=3  done
  [16/36] MISSILES  → part_n_extraterritorial   L=1  done
  [17/36] MISSILES  → part_n_ext

## Section 5 — Visualization

Three figures:
1. **CCF heatmap grid** — one heatmap per outcome; rows = weapon class, columns = lag, color = mean ρ
2. **DH Z-statistic forest plot** — coefficients across (weapon, outcome, lag) cells
3. **Headline trajectory** — Turkey post-2016 (Bayraktar export buildup) as the canonical Drone Effect case

In [6]:
# ── Fig 1: CCF heatmap grid (one panel per outcome) ───────────────────────────
fig, axes = plt.subplots(1, len(OUTCOMES), figsize=(18, 4.5), sharey=True)
for ax, outcome in zip(axes, OUTCOMES):
    sub = ccf_mean[ccf_mean["outcome"] == outcome]
    hm = sub.pivot(index="weapon", columns="lag", values="rho_mean")
    # Reorder weapons consistently
    hm = hm.reindex(list(WEAPON_CLASSES))
    sns.heatmap(hm, annot=True, fmt=".2f", center=0, cmap="RdBu_r",
                vmin=-0.15, vmax=0.15, ax=ax, cbar_kws={"label": "mean ρ"}, square=False)
    ax.set_title(outcome, fontsize=10)
    ax.set_xlabel("Lag (years; positive = treatment leads outcome)")
    ax.set_ylabel("Weapon class" if ax is axes[0] else "")
fig.suptitle("Cross-correlation functions: TIV (Δlog) → conflict outcome (Δlog)", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_ccf_heatmaps.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print("Fig 1 saved.")

# ── Fig 2: DH Z-statistic forest plot ─────────────────────────────────────────
fig, axes = plt.subplots(1, len(OUTCOMES), figsize=(18, 5), sharey=False)
lags_plot = [1, 2, 3]
for ax, outcome in zip(axes, OUTCOMES):
    sub = dh_df[dh_df["outcome"] == outcome].copy()
    weapons = list(WEAPON_CLASSES)
    width = 0.25
    x = np.arange(len(weapons))
    for i, lag in enumerate(lags_plot):
        zs = [sub[(sub["weapon"] == w) & (sub["lag"] == lag)]["Z"].iloc[0]
              if not sub[(sub["weapon"] == w) & (sub["lag"] == lag)].empty else 0
              for w in weapons]
        ax.bar(x + (i - 1) * width, zs, width, label=f"L={lag}")
    ax.axhline(0, color="gray", lw=0.7)
    ax.axhline(1.645, color="firebrick", lw=0.7, linestyle="--", alpha=0.6, label="Z = 1.645 (p=0.05)")
    ax.set_xticks(x)
    ax.set_xticklabels(weapons, fontsize=9)
    ax.set_title(outcome, fontsize=10)
    ax.set_ylabel("Dumitrescu-Hurlin Z")
    ax.legend(fontsize=7)
fig.suptitle("Panel Granger causality (Dumitrescu-Hurlin): treatment leads outcome", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_dh_forest.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print("Fig 2 saved.")

# ── Fig 3: Headline Turkey trajectory ────────────────────────────────────────
headline_iso = "TUR"
tur = panel[panel["iso3"] == headline_iso].sort_values("year")
if not tur.empty:
    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.plot(tur["year"], tur["log_tiv_AIR"], color="firebrick", lw=2,
             label="log(1 + TIV AIR imports)")
    ax1.set_xlabel("Year")
    ax1.set_ylabel("TIV (AIR class, log1p)", color="firebrick")
    ax1.tick_params(axis="y", labelcolor="firebrick")
    ax2 = ax1.twinx()
    ax2.plot(tur["year"], tur["log_part_n_minor"], color="steelblue", lw=2,
             label="log(1 + minor-conflict participations)")
    ax2.set_ylabel("Minor conflicts (log1p)", color="steelblue")
    ax2.tick_params(axis="y", labelcolor="steelblue")
    ax1.set_title(f"{headline_iso}: AIR imports vs minor-conflict participation, {ANALYSIS_YEAR_MIN}–{ANALYSIS_YEAR_MAX}")
    ax1.spines["top"].set_visible(False)
    ax2.spines["top"].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig3_turkey_trajectory.png", dpi=300)
    plt.close(fig)
    print("Fig 3 saved.")

Fig 1 saved.
Fig 2 saved.
Fig 3 saved.


## Section 6 — Sanity Checks

In [7]:
results = []
def chk(label, expr):
    status = "PASS" if expr else "FAIL"
    results.append((label, status))
    print(f"[{status}] {label}")

# [1] Panel built correctly
chk("[1] Panel shape sensible (>2000 rows, <10000)",
    2000 < panel.shape[0] < 10000)

# [2] All 4 weapon-class columns built
chk("[2] All 4 weapon classes constructed",
    all(f"log_tiv_{c}" in panel.columns for c in WEAPON_CLASSES))

# [3] CCF grid: 4 weapons × 3 outcomes × 6 lags = 72 (weapon, outcome, lag) cells
chk("[3] CCF aggregate has 72 cells (4 × 3 × 6)",
    len(ccf_mean) == 72)

# [4] DH grid: 4 weapons × 3 outcomes × 3 lags = 36 cells
chk("[4] DH table has 36 cells (4 × 3 × 3)",
    len(dh_df) == 36)

# [5] DH country coverage adequate (≥ 50 countries in main test)
min_n_countries = dh_df["n_countries"].min()
chk(f"[5] DH n_countries ≥ 50 in every cell (min: {min_n_countries})",
    min_n_countries >= 50)

# [6] Placebo grid same size
chk("[6] Placebo table has 36 cells", len(placebo_df) == 36)

# [7] Placebo rejection rate near nominal α (within reason)
mean_placebo_rate = placebo_df["placebo_sig_rate"].mean()
chk(f"[7] Mean placebo rejection rate < 0.20 (got {mean_placebo_rate:.3f})",
    mean_placebo_rate < 0.20)

# [8] All 5 result tables saved
expected_tables = [
    "section1_stationarity_audit.csv",
    "section2_ccf_long.csv", "section2_ccf_aggregated.csv",
    "section3_dumitrescu_hurlin.csv",
    "section4_placebo_comparison.csv",
]
chk("[8] All result tables saved",
    all((TBL_DIR / t).exists() for t in expected_tables))

# [9] All 3 figures saved
expected_figs = ["fig1_ccf_heatmaps.png", "fig2_dh_forest.png", "fig3_turkey_trajectory.png"]
chk("[9] All 3 figures saved",
    all((FIG_DIR / f).exists() for f in expected_figs))

n_pass = sum(1 for _, r in results if r == "PASS")
n_fail = sum(1 for _, r in results if r == "FAIL")
print(f"\n{n_pass}/{len(results)} checks passed, {n_fail} failed")

[PASS] [1] Panel shape sensible (>2000 rows, <10000)
[PASS] [2] All 4 weapon classes constructed
[PASS] [3] CCF aggregate has 72 cells (4 × 3 × 6)
[PASS] [4] DH table has 36 cells (4 × 3 × 3)
[PASS] [5] DH n_countries ≥ 50 in every cell (min: 87)
[PASS] [6] Placebo table has 36 cells
[PASS] [7] Mean placebo rejection rate < 0.20 (got 0.116)
[PASS] [8] All result tables saved
[PASS] [9] All 3 figures saved

9/9 checks passed, 0 failed


## Section 7 — Headline Findings

In [8]:
print("=" * 70)
print("NB-06 RQ2 HEADLINE FINDINGS")
print("=" * 70)
print()
print("Treatment: SIPRI TIV deliveries (4 weapon classes, log1p, first-differenced)")
print("Outcome:   UCDP participation counts (3 outcomes, log1p, first-differenced)")
print(f"Sample:    {panel['iso3'].nunique()} countries × {panel['year'].nunique()} years")
print(f"Method:    Cross-correlation + Dumitrescu-Hurlin (2012) panel Granger")
print()
print("Drone Effect hypothesis predictions:")
print("  AIR/MISSILES → part_n_minor:           positive Z, p < 0.05  (substitution)")
print("  AIR/MISSILES → part_n_war:             null  Z, p > 0.10     (deterrence)")
print("  AIR/MISSILES → part_n_extraterritorial: positive Z, p < 0.05 (force projection)")
print()
print("Realized results — significant Z (p < 0.05):")
sig = dh_df[dh_df["p"] < 0.05].sort_values(["weapon", "outcome", "lag"])
if len(sig) == 0:
    print("  (None — no Granger-causal precedence detected at p < 0.05)")
else:
    for _, row in sig.iterrows():
        # Check the placebo column too
        pl = placebo_df[(placebo_df["weapon"] == row["weapon"]) &
                        (placebo_df["outcome"] == row["outcome"]) &
                        (placebo_df["lag"] == row["lag"])]
        pl_rate = pl["placebo_sig_rate"].iloc[0] if not pl.empty else np.nan
        verdict = "TRUSTWORTHY" if pl_rate < 0.15 else "FLAGGED (placebo high)"
        print(f"  {row['weapon']:9s} → {row['outcome']:25s} L={int(row['lag'])} "
              f"Z={row['Z']:+.2f} p={row['p']:.4f} placebo={pl_rate:.2f} [{verdict}]")

print()
print(f"Outputs:")
print(f"  Tables  → {TBL_DIR}/")
print(f"  Figures → {FIG_DIR}/")
print()
print("=== NB-06 complete — proceed to NB-07 (RQ3 clustering with bootstrap) ===")

NB-06 RQ2 HEADLINE FINDINGS

Treatment: SIPRI TIV deliveries (4 weapon classes, log1p, first-differenced)
Outcome:   UCDP participation counts (3 outcomes, log1p, first-differenced)
Sample:    192 countries × 36 years
Method:    Cross-correlation + Dumitrescu-Hurlin (2012) panel Granger

Drone Effect hypothesis predictions:
  AIR/MISSILES → part_n_minor:           positive Z, p < 0.05  (substitution)
  AIR/MISSILES → part_n_war:             null  Z, p > 0.10     (deterrence)
  AIR/MISSILES → part_n_extraterritorial: positive Z, p < 0.05 (force projection)

Realized results — significant Z (p < 0.05):
  GROUND    → part_n_extraterritorial   L=1 Z=+4.93 p=0.0000 placebo=0.34 [FLAGGED (placebo high)]
  GROUND    → part_n_minor              L=1 Z=+1.87 p=0.0308 placebo=0.28 [FLAGGED (placebo high)]
  GROUND    → part_n_war                L=1 Z=+6.42 p=0.0000 placebo=0.36 [FLAGGED (placebo high)]
  MISSILES  → part_n_extraterritorial   L=1 Z=+2.02 p=0.0218 placebo=0.48 [FLAGGED (placebo hig

## Section 8 — Interpreting the Null Result for the Drone Effect

Zero (weapon, outcome, lag) cells survive the placebo falsification test. All 8 cells that
were significant at p < 0.05 in the raw Dumitrescu-Hurlin test show placebo rejection rates
of 0.20–0.50 — far above the 0.15 trustworthiness threshold — meaning the same "significant"
result appears almost as often on randomly shuffled data as on the real series. This pattern
is consistent with residual autocorrelation in the differenced series rather than genuine
lead-lag structure.

Most notably, the AIR weapon class — aircraft and drones, the class the Drone Effect
hypothesis is built around — shows no significant DH result at any lag for any outcome.
The hypothesis that precision-strike/air capability acquisitions precede a rise in
low-intensity conflict within a short window is not supported by this panel.

**Implication for the writeup.** RQ2 should be reported as a genuine null: the
cross-correlation patterns (Section 2) are weak and inconsistent in sign, and the one
analytical method built to test causal precedence rigorously (DH + placebo) finds nothing
that survives falsification. This is a legitimate, reportable finding — the absence of
lead-lag structure is itself informative about how weapon acquisitions and conflict onset
relate (or fail to relate) at the country-year level.